### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="automobile",
    dataset_year="1985",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5B01C",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/automobile/ && wget -P local-data-warehouse/automobile/ https://archive.ics.uci.edu/static/public/10/automobile.zip && unzip local-data-warehouse/automobile/automobile.zip -d local-data-warehouse/automobile/ && rm local-data-warehouse/automobile/automobile.zip && mv local-data-warehouse/automobile/automobile/* local-data-warehouse/automobile/ && rm -r local-data-warehouse/automobile/automobile
""",
    # References
    academic_reference_bibtex="""@misc{Schlimmer1985Automobile,
  title={Automobile},
  author={Jeffrey Schlimmer},
  year={1985},
  title={1985 Model Import Car and Truck Specifications},
  journal={Ward's Automotive Yearbook}
}
""",
    academic_reference_bibtex_key="Schlimmer1985Automobile",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
        - We rename the target feature from "symboling" to "risk_factor" for clarity of the task.
        - We remove "normalized-losses" attribute as it has been handcrafted and might lead to data leaks.
        - We encode missing values as np.nan instead of "?".
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="risk_factor",
    problem_type="regression",
    objective_metric_name="rmse",
    stratify_on="risk_factor",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

column_names = [
"risk_factor",
"normalized-losses",
"make",
"fuel-type",
"aspiration",
"num-of-doors",
"body-style",
"drive-wheels",
"engine-location",
"wheel-base",
"length",
"width",
"height",
"curb-weight",
"engine-type",
"num-of-cylinders",
"engine-size",
"fuel-system",
"bore",
"stroke",
"compression-ratio",
"horsepower",
"peak-rpm",
"city-mpg",
"highway-mpg",
"price",
]

df = pd.read_csv(dataset_mold.path / "imports-85.data", header=None, index_col=False, names=column_names)
df.replace("?", np.nan, inplace=True)

cat_columns = ["risk_factor", "make", "fuel-type", "aspiration", "num-of-doors", "body-style", "drive-wheels", "engine-location", "engine-type", "num-of-cylinders", "fuel-system"]
df[cat_columns] = df[cat_columns].astype("category")

non_cat_cols = df.columns[~df.dtypes.eq("category")]
df[non_cat_cols] = df[non_cat_cols].apply(pd.to_numeric, errors="coerce")

df = df.drop("normalized-losses", axis=1)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 205
Columns: 25
Use sampling: False (sample size: 205)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['price', 'curb-weight', 'length', 'horsepower', 'wheel-base', 'height', 'engine-size', 'width', 'bore', 'stroke']
Rows remaining as candidates after top-10 filter: 6 (of 205)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


/Users/schaefer.bastian/miniconda3/envs/jupyter_env-py3-10/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/schaefer.bastian/miniconda3/envs/jupyter_env-py3-10/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/schaefer.bastian/miniconda3/envs/jupyter_env-py3-10/lib/python3.10/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [4]:
# Sample Rows
df_head

,risk_factor,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,length,width,height,curb-weight,engine-type,num-of-cylinders,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0
1,3,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0
2,1,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500.0
3,2,audi,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950.0
4,2,audi,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450.0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,num-of-doors,category,2.0,0.98,2.0,"four, two"
1,risk_factor,category,0.0,0.00,6.0,"0, 1, 2, 3, -1, -2"
2,make,category,0.0,0.00,22.0,"toyota, nissan, mazda, mitsubishi, honda, volkswagen, subaru, peugot, volvo, dodge"
3,fuel-type,category,0.0,0.00,2.0,"gas, diesel"
4,aspiration,category,0.0,0.00,2.0,"std, turbo"
5,body-style,category,0.0,0.00,5.0,"sedan, hatchback, wagon, hardtop, convertible"
6,drive-wheels,category,0.0,0.00,3.0,"fwd, rwd, 4wd"
7,engine-location,category,0.0,0.00,2.0,"front, rear"
8,engine-type,category,0.0,0.00,7.0,"ohc, ohcf, ohcv, dohc, l, rotor, dohcv"
9,num-of-cylinders,category,0.0,0.00,7.0,"four, six, five, eight, two, three, twelve"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
wheel-base,205.0,98.756585,6.021776,86.60,120.90
length,205.0,174.049268,12.337289,141.10,208.10
width,205.0,65.907805,2.145204,60.30,72.30
height,205.0,53.724878,2.443522,47.80,59.80
curb-weight,205.0,2555.565854,520.680204,1488.00,4066.00
engine-size,205.0,126.907317,41.642693,61.00,326.00
bore,201.0,3.329751,0.273539,2.54,3.94
stroke,201.0,3.255423,0.316717,2.07,4.17
compression-ratio,205.0,10.142537,3.972040,7.00,23.00
horsepower,203.0,104.256158,39.714369,48.00,288.00


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column           rank                           
aspiration       1             std    168  81.95
                 2           turbo     37  18.05
body-style       1           sedan     96  46.83
                 2       hatchback     70  34.15
                 3           wagon     25  12.20
                 4         hardtop      8   3.90
                 5     convertible      6   2.93
drive-wheels     1             fwd    120  58.54
                 2             rwd     76  37.07
                 3             4wd      9   4.39
engine-location  1           front    202  98.54
                 2            rear      3   1.46
engine-type      1             ohc    148  72.20
                 2            ohcf     15   7.32
                 3            ohcv     13   6.34
                 4            dohc     12   5.85
                 5               l     12   5.85
fuel-system      1            mpfi     94  45.85
                 2            2bbl     66  32.20
                 3             idi     20   9.76
                 4            1bbl     11   5.37
                 5            spdi      9   4.39
fuel-type        1             gas    185  90.24
                 2          diesel     20   9.76
make             1          toyota     32  15.61
                 2          nissan     18   8.78
                 3           mazda     17   8.29
                 4      mitsubishi     13   6.34
                 5           honda     13   6.34
num-of-cylinders 1            four    159  77.56
                 2             six     24  11.71
                 3            five     11   5.37
                 4           eight      5   2.44
                 5             two      4   1.95
num-of-doors     1            four    114  55.61
                 2             two     89  43.41
                 3            <NA>      2   0.98
risk_factor      1               0     67  32.68
                 2               1     54  26.34
                 3               2     32  15.61
                 4               3     27  13.17
                 5              -1     22  10.73

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,44.88,0.211,NaN,1.551,NaN,log1p,463.2,1075.8,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to automobile/019d9cbf-79e2-7acb-aead-892b101944b6
019d9cbf-79e2-7acb-aead-892b101944b6
e3086f542fd98647eb3c0181b7b0a87b1d40f797d7939ac8ee0eeab481369316
